In [17]:
import mysql.connector
from mysql.connector import Error

# --- DB 연결 정보 ---
DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': '0000',
    'database': 'deep_dish'
}

BATCH_SIZE = 10000

def sync_timestamps_in_batches():
    db_conn = None
    cursor = None
    updated_total = 0
    
    print("데이터 동기화 작업을 시작합니다...")

    try:
        db_conn = mysql.connector.connect(**DB_CONFIG)
        
        while True:
            cursor = db_conn.cursor(dictionary=True)

            query_select = f"""
                SELECT
                    ar.result_id,
                    i.created_at
                FROM
                    analysis_results ar
                JOIN
                    images i ON ar.image_id = i.image_id
                WHERE
                    ar.created_at != i.created_at
                LIMIT {BATCH_SIZE};
            """
            cursor.execute(query_select)
            records_to_update = cursor.fetchall()

            if not records_to_update:
                print("\n모든 데이터 동기화가 완료되었습니다.")
                break

            update_data_tuples = [
                (record['created_at'], record['result_id'])
                for record in records_to_update
            ]

            query_update = "UPDATE analysis_results SET created_at = %s WHERE result_id = %s"
            cursor.executemany(query_update, update_data_tuples)
            db_conn.commit()

            updated_count = cursor.rowcount
            updated_total += updated_count
            print(f"{updated_count}개 행 업데이트 완료. (총 {updated_total}개 처리)")

            cursor.close()

    except Error as e:
        print(f"DB 작업 중 에러 발생: {e}")
        if db_conn:
            db_conn.rollback()
    finally:
        # === 수정된 부분 ===
        # cursor.is_closed() 라는 불필요한 확인 제거
        if db_conn and db_conn.is_connected():
            if cursor: # cursor 객체가 생성되었는지 확인
                cursor.close()
            db_conn.close()
            print("MySQL 연결이 해제되었습니다.")

# --- 스크립트 실행 ---
if __name__ == '__main__':
    sync_timestamps_in_batches()

데이터 동기화 작업을 시작합니다...

모든 데이터 동기화가 완료되었습니다.
MySQL 연결이 해제되었습니다.


In [14]:
# app.py
from flask import Flask, render_template, request
import mysql.connector
import os # os 모듈 추가
from dotenv import load_dotenv # dotenv 모듈 추가

load_dotenv() # .env 파일에서 환경 변수를 불러옴

app = Flask(__name__)

# --- DB 연결 설정 (수정된 부분) ---
DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME')
}
db_conn = mysql.connector.connect(**DB_CONFIG)
cursor = db_conn.cursor(dictionary=True) # 결과를 딕셔너리로 받기
cursor.execute("SELECT * FROM analysis_results limit 10;")

food_trends = cursor.fetchall() # 쿼리 결과를 가져옴
food_trends

[{'result_id': 1,
  'image_id': 1,
  'result_type': 'FOOD',
  'detected_id': 1,
  'confidence_score': 0.6086,
  'created_at': datetime.datetime(2025, 7, 23, 14, 41, 24)},
 {'result_id': 2,
  'image_id': 2,
  'result_type': 'FOOD',
  'detected_id': 1,
  'confidence_score': 0.6428,
  'created_at': datetime.datetime(2025, 7, 23, 14, 41, 24)},
 {'result_id': 3,
  'image_id': 38582,
  'result_type': 'FOOD',
  'detected_id': 1,
  'confidence_score': 0.6485,
  'created_at': datetime.datetime(2025, 7, 23, 14, 41, 24)},
 {'result_id': 4,
  'image_id': 4,
  'result_type': 'FOOD',
  'detected_id': 1,
  'confidence_score': 0.6921,
  'created_at': datetime.datetime(2025, 7, 23, 14, 41, 24)},
 {'result_id': 5,
  'image_id': 5,
  'result_type': 'FOOD',
  'detected_id': 1,
  'confidence_score': 0.689,
  'created_at': datetime.datetime(2025, 7, 23, 14, 41, 24)},
 {'result_id': 6,
  'image_id': 6,
  'result_type': 'FOOD',
  'detected_id': 1,
  'confidence_score': 0.6273,
  'created_at': datetime.datetim